# Gov Collect Agents

Three sources, one table, in precedence order:

1. **Agent 365 registry** — widest view, but **preview**, needs AI
   Administrator, and does not show draft agents outside Copilot Studio.
2. **Entra Agent ID** — identities, blueprints, and the mandatory human sponsor.
3. **Dataverse `bot` table** — the Copilot Studio truth including drafts, and
   the **licence-free fallback** when the tenant has no Agent 365 entitlement.

The merge is where the value is: an agent seen *only* by the tenant-wide
registry and by none of the sources we govern is a **shadow agent**, and an
agent with neither owner nor sponsor is `is_ownerless`. Both are Critical
findings, and neither is discoverable from a single source.

**Agent 365 does not gate creation.** Everything here is post-hoc inventory;
the only preventive, class-level lever is the blueprint, which arrives as an
actuator in a later phase.

In [ ]:
dry_run = True
lakehouse_name = "governance_lh"
# When false, skip the registry entirely — the documented degraded path for a
# tenant without an Agent 365 licence.
use_agent365_registry = True

In [ ]:
# --- inlined from collectors/shape_common.py (unit-tested offline) ---
from __future__ import annotations

import json
import uuid
from datetime import datetime, timezone
from typing import Any, Callable, Iterable, Sequence


def utcnow() -> datetime:
    return datetime.now(timezone.utc)


def new_run_id() -> str:
    return str(uuid.uuid4())


def as_str(value: Any) -> str | None:
    """Normalise an API scalar to a string, preserving a real absence as None.

    Collector tables are all-string on purpose: every plane has its own id
    format, and coercing them into typed columns is how a join silently starts
    returning nothing.
    """
    if value is None:
        return None
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def as_json(value: Any) -> str | None:
    """Stable JSON for a blob column. Sorted keys so diffs are meaningful."""
    if value is None:
        return None
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)


def stamp(rows: Iterable[dict], run_id: str, scanned_at: datetime | None = None) -> list[dict]:
    """Attach the run provenance every `gov_actual_*` row carries."""
    when = scanned_at or utcnow()
    out = []
    for row in rows:
        enriched = dict(row)
        enriched["run_id"] = run_id
        enriched["scanned_at"] = when
        out.append(enriched)
    return out


class RunLedger:
    """Accumulates what a collector did, for the `gov_runs` row and the app.

    Errors are first-class: a collector that quietly drops an unreadable object
    produces a governance report that is wrong in the most dangerous direction —
    it under-reports access.
    """

    def __init__(self, collector: str, module: str, tier: str) -> None:
        self.run_id = new_run_id()
        self.collector = collector
        self.module = module
        self.tier = tier
        self.started_at = utcnow()
        self.finished_at: datetime | None = None
        self.errors: list[dict[str, str]] = []
        self.counts: dict[str, int] = {}

    def count(self, table: str, n: int) -> None:
        self.counts[table] = self.counts.get(table, 0) + n

    def error(self, scope: str, exc: BaseException | str) -> None:
        self.errors.append(
            {
                "scope": scope,
                "type": type(exc).__name__ if isinstance(exc, BaseException) else "Error",
                "message": str(exc),
            }
        )

    def finish(self) -> dict:
        self.finished_at = utcnow()
        return {
            "run_id": self.run_id,
            "collector": self.collector,
            "module": self.module,
            "tier": self.tier,
            "started_at": self.started_at,
            "finished_at": self.finished_at,
            "n_objects": sum(self.counts.values()),
            "n_errors": len(self.errors),
            "error_json": as_json(self.errors) if self.errors else None,
            "duration_s": (self.finished_at - self.started_at).total_seconds(),
        }

    def exit_value(self, *, dry_run: bool) -> dict:
        """Actuator-contract-shaped result (PLAN.md §14) for the app to parse."""
        return {
            "ok": True,
            "dry_run": dry_run,
            "run_id": self.run_id,
            "collector": self.collector,
            "module": self.module,
            "tier": self.tier,
            "counts": dict(self.counts),
            "n_errors": len(self.errors),
            "errors": self.errors[:20],
            "finished_at": (self.finished_at or utcnow()).isoformat(),
        }


def safe_each(
    items: Sequence[Any],
    fn: Callable[[Any], list[dict]],
    ledger: RunLedger,
    scope_of: Callable[[Any], str],
) -> list[dict]:
    """Map `fn` over `items`, recording per-item failures instead of raising."""
    rows: list[dict] = []
    for item in items:
        try:
            rows.extend(fn(item))
        except Exception as exc:  # noqa: BLE001 — a collector must never hard-fail
            ledger.error(scope_of(item), exc)
    return rows

In [ ]:
# --- inlined from collectors/runtime.py (unit-tested offline) ---
from __future__ import annotations

import json
import time
from typing import Any, Callable


class RestError(RuntimeError):
    def __init__(self, status: int, url: str, body: str) -> None:
        super().__init__(f"{status} {url}: {body[:400]}")
        self.status = status
        self.url = url


def fabric_client():
    """A `sempy` REST client for Fabric / Power BI, under the running identity."""
    import sempy.fabric as fabric  # type: ignore

    return fabric.FabricRestClient()


def rest_get(client, path: str, *, retries: int = 4) -> dict[str, Any]:
    """GET with backoff on 429/5xx.

    Admin APIs are rate-limited (25 req/min on some tenant-setting endpoints), and
    a nightly crawl that gives up on the first 429 silently under-reports — which
    is the worst possible failure mode for a governance inventory.
    """
    delay = 2.0
    last: Exception | None = None
    for _ in range(retries):
        response = client.get(path)
        if response.status_code == 200:
            return response.json() if response.text else {}
        if response.status_code in (429, 500, 502, 503, 504):
            retry_after = response.headers.get("Retry-After")
            time.sleep(float(retry_after) if retry_after else delay)
            delay = min(delay * 2, 60)
            last = RestError(response.status_code, path, response.text)
            continue
        raise RestError(response.status_code, path, response.text)
    raise last or RestError(0, path, "exhausted retries")


def graph_token(scope: str = "https://graph.microsoft.com/.default") -> str:
    """Delegated Graph token for the identity the notebook runs as."""
    import notebookutils  # type: ignore

    return notebookutils.credentials.getToken(scope)


def graph_get(token: str, url: str, *, retries: int = 4) -> dict[str, Any]:
    import urllib.error
    import urllib.request

    if not url.startswith("http"):
        url = f"https://graph.microsoft.com{url}"

    delay = 2.0
    for _ in range(retries):
        request = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
        try:
            with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            if exc.code in (429, 500, 502, 503, 504):
                time.sleep(delay)
                delay = min(delay * 2, 60)
                continue
            raise RestError(exc.code, url, exc.read().decode("utf-8", "replace")) from exc
    raise RestError(0, url, "exhausted retries")


def graph_call(token: str, method: str, url: str, body: dict | None = None) -> dict[str, Any]:
    """Graph request with a method — the write-capable sibling of `graph_get`.

    Deliberately **not** retried on 5xx: a POST that may have partially applied
    must not be replayed blindly. The actuator's read-before-write makes a
    retry safe only after re-reading, and that is the caller's decision.
    """
    import urllib.error
    import urllib.request

    if not url.startswith("http"):
        url = f"https://graph.microsoft.com{url}"

    data = json.dumps(body).encode("utf-8") if body is not None else None
    request = urllib.request.Request(url, data=data, method=method.upper())
    request.add_header("Authorization", f"Bearer {token}")
    if data is not None:
        request.add_header("Content-Type", "application/json")

    try:
        with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
            payload = response.read().decode("utf-8")
            return json.loads(payload) if payload else {}
    except urllib.error.HTTPError as exc:
        raise RestError(exc.code, url, exc.read().decode("utf-8", "replace")) from exc


def fabric_call(client, method: str, path: str, body: dict | None = None) -> dict[str, Any]:
    """Fabric REST with a method, through the `sempy` client.

    Same no-retry stance as `graph_call`, for the same reason.
    """
    verb = method.upper()
    if verb == "GET":
        response = client.get(path)
    elif verb == "POST":
        response = client.post(path, json=body or {})
    elif verb == "PATCH":
        response = client.patch(path, json=body or {})
    elif verb == "DELETE":
        response = client.delete(path)
    else:
        raise ValueError(f"unsupported method {method}")

    if response.status_code not in (200, 201, 202, 204):
        raise RestError(response.status_code, path, response.text)
    return response.json() if response.text else {}


def write_table(
    spark,
    lakehouse: str,
    table: str,
    rows: list[dict],
    *,
    dry_run: bool,
    log: Callable[[str, str, str], None],
) -> int:
    """Overwrite one `gov_actual_*` table with this run's rows.

    Overwrite, not append: these tables are a *snapshot of current reality*, and
    the run ledger plus `gov_audit` carry the history. An append-only actual-state
    table is how a drift engine starts comparing against last month.
    """
    if dry_run:
        log(table, "Planned", f"{len(rows)} rows")
        return len(rows)
    if not rows:
        log(table, "Skipped (no permission)", "no rows collected")
        return 0
    try:
        df = spark.createDataFrame(rows)
        df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
            f"{lakehouse}.{table}"
        )
        log(table, "Created", f"{len(rows)} rows")
        return len(rows)
    except Exception as exc:  # noqa: BLE001
        log(table, "Failed", f"{type(exc).__name__}: {exc}")
        return 0


def write_run_row(spark, lakehouse: str, summary: dict, *, dry_run: bool) -> None:
    if dry_run:
        return
    try:
        spark.createDataFrame([summary]).write.mode("append").option(
            "mergeSchema", "true"
        ).saveAsTable(f"{lakehouse}.gov_runs")
    except Exception as exc:  # noqa: BLE001
        print(f"gov_runs append failed: {exc}")


def finish(ledger, spark, lakehouse: str, *, dry_run: bool) -> str:
    summary = ledger.finish()
    write_run_row(spark, lakehouse, summary, dry_run=dry_run)
    result = ledger.exit_value(dry_run=dry_run)
    try:
        import notebookutils  # type: ignore

        notebookutils.notebook.exit(json.dumps(result))
    except ImportError:
        print(json.dumps(result, indent=2))
    return json.dumps(result)

In [ ]:
# --- inlined from collectors/shape_agent.py (unit-tested offline) ---
from __future__ import annotations

from typing import Any, Iterable



#: Registry platform strings → our normalised platform.
PLATFORM_MAP = {
    "mcs": "CopilotStudio",
    "copilotstudio": "CopilotStudio",
    "agentbuilder": "AgentBuilder",
    "declarative": "AgentBuilder",
    "sharepoint": "SharePoint",
    "foundry": "Foundry",
    "toolkit": "ToolkitSDK",
}

#: Third-party platforms Registry sync supports today. Anything else that shows
#: up is unmanaged by definition.
SYNCED_THIRD_PARTY = {"bedrock", "vertex", "agentforce", "genie"}


def normalise_platform(raw: str | None) -> str:
    if not raw:
        return "Unknown"
    value = raw.strip().lower().replace(" ", "").replace("-", "")
    for needle, platform in PLATFORM_MAP.items():
        if needle in value:
            return platform
    for needle in SYNCED_THIRD_PARTY:
        if needle in value:
            return "ThirdParty"
    return "Unknown"


def shape_registry_agents(payload: dict[str, Any]) -> list[dict]:
    """Agent 365 registry (`List Copilot packages`, preview)."""
    rows: list[dict] = []
    for pkg in payload.get("value", []) or []:
        owner = pkg.get("owner") or {}
        rows.append(
            {
                "agent_id": as_str(pkg.get("id")),
                "name": as_str(pkg.get("displayName") or pkg.get("name")),
                "platform": normalise_platform(
                    as_str(pkg.get("platform") or pkg.get("agentType"))
                ),
                "source": "A365Registry",
                "state": as_str(pkg.get("state") or pkg.get("publishingState")),
                "owner_principal": as_str(
                    owner.get("id") or owner.get("userPrincipalName") or pkg.get("ownerId")
                ),
                "sponsor_principal": None,
                "blueprint_id": None,
                "agent_identity_id": as_str(pkg.get("agentIdentityId")),
                "environment_id": as_str(pkg.get("environmentId")),
                "risk_flags_json": as_json(pkg.get("risks")),
                "created_at": as_str(pkg.get("createdDateTime")),
            }
        )
    return rows


def shape_agent_identities(payload: dict[str, Any]) -> list[dict]:
    """Entra `agentIdentity` service principals.

    Every agent identity requires a human sponsor; if the sponsor leaves,
    sponsorship transfers to their manager. A missing sponsor is therefore not a
    data-quality problem, it is a governance finding.
    """
    rows: list[dict] = []
    for identity in payload.get("value", []) or []:
        sponsors = identity.get("sponsors") or []
        sponsor = sponsors[0] if sponsors else identity.get("sponsor")
        rows.append(
            {
                "agent_id": as_str(identity.get("id")),
                "name": as_str(identity.get("displayName")),
                "platform": normalise_platform(as_str(identity.get("agentIdentityType"))),
                "source": "EntraAgentID",
                "state": as_str(
                    "Disabled" if identity.get("accountEnabled") is False else "Published"
                ),
                "owner_principal": as_str((identity.get("owners") or [{}])[0].get("id"))
                if identity.get("owners")
                else None,
                "sponsor_principal": as_str(
                    sponsor.get("id") if isinstance(sponsor, dict) else sponsor
                ),
                "blueprint_id": as_str(identity.get("agentIdentityBlueprintId")),
                "agent_identity_id": as_str(identity.get("id")),
                "environment_id": None,
                "risk_flags_json": None,
                "created_at": as_str(identity.get("createdDateTime")),
            }
        )
    return rows


def shape_blueprints(payload: dict[str, Any]) -> list[dict]:
    rows: list[dict] = []
    for blueprint in payload.get("value", []) or []:
        name = as_str(blueprint.get("displayName")) or ""
        rows.append(
            {
                "blueprint_id": as_str(blueprint.get("id")),
                "display_name": name,
                "is_multitenant": as_str(
                    blueprint.get("signInAudience") not in (None, "AzureADMyOrg")
                ),
                "sponsor_principal": as_str(
                    ((blueprint.get("sponsors") or [{}])[0] or {}).get("id")
                ),
                "granted_permissions_json": as_json(
                    blueprint.get("requiredResourceAccess")
                ),
                "is_app_managed": as_str(name.startswith("GOV-")),
            }
        )
    return rows


def shape_dataverse_bots(environment_id: str, payload: dict[str, Any]) -> list[dict]:
    """Copilot Studio agents from the Dataverse `bot` table.

    This is the **licence-free fallback**: it needs no Agent 365 entitlement, and
    it is the only source that sees *draft* agents.
    """
    rows: list[dict] = []
    for bot in payload.get("value", []) or []:
        published = bot.get("publishedon")
        rows.append(
            {
                "agent_id": as_str(bot.get("botid")),
                "name": as_str(bot.get("name") or bot.get("schemaname")),
                "platform": "CopilotStudio",
                "source": "Dataverse",
                "state": "Published" if published else "Draft",
                "owner_principal": as_str(bot.get("_ownerid_value")),
                "sponsor_principal": None,
                "blueprint_id": None,
                "agent_identity_id": None,
                "environment_id": as_str(environment_id),
                "risk_flags_json": None,
                "created_at": as_str(bot.get("createdon")),
            }
        )
    return rows


#: Source precedence when the same agent is seen more than once. The registry
#: has the widest metadata; Dataverse is authoritative for Copilot Studio state
#: (it is the only source that distinguishes Draft).
_SOURCE_RANK = {"A365Registry": 3, "EntraAgentID": 2, "Dataverse": 1}


def _key(row: dict) -> str:
    return (
        as_str(row.get("agent_identity_id"))
        or as_str(row.get("agent_id"))
        or f"name:{as_str(row.get('name'))}"
    )


def merge_agents(*sources: Iterable[dict]) -> list[dict]:
    """Merge agent rows from every source into one deduplicated inventory.

    Rules:
      * merge on agent identity id when present, else agent id, else name
      * higher-ranked sources win on conflicting scalars, but a **non-empty**
        value always beats an empty one — a sponsor known only to Entra must not
        be erased by a registry row that omits it
      * `sources_json` records every source that saw the agent, which is what
        makes shadow detection possible
    """
    merged: dict[str, dict] = {}
    seen_sources: dict[str, set[str]] = {}

    for source in sources:
        for row in source:
            key = _key(row)
            seen_sources.setdefault(key, set()).add(str(row.get("source")))
            existing = merged.get(key)
            if existing is None:
                merged[key] = dict(row)
                continue

            incoming_rank = _SOURCE_RANK.get(str(row.get("source")), 0)
            existing_rank = _SOURCE_RANK.get(str(existing.get("source")), 0)
            for field, value in row.items():
                if value in (None, ""):
                    continue
                if existing.get(field) in (None, "") or incoming_rank > existing_rank:
                    existing[field] = value
            if incoming_rank > existing_rank:
                existing["source"] = row.get("source")

    out: list[dict] = []
    for key, row in merged.items():
        sources_seen = sorted(seen_sources.get(key, set()))
        owner = row.get("owner_principal")
        sponsor = row.get("sponsor_principal")
        row["sources_json"] = as_json(sources_seen)
        # Seen only by the tenant-wide registry and by none of the sources we
        # actually govern → nobody here provisioned it.
        row["is_shadow"] = as_str(sources_seen == ["A365Registry"])
        row["is_ownerless"] = as_str(not owner and not sponsor)
        out.append(row)

    return sorted(out, key=lambda r: (str(r.get("name") or ""), str(r.get("agent_id") or "")))

In [ ]:
steps = []


def log(step, status, detail=""):
    steps.append({"step": step, "status": status, "detail": detail})
    print(f"[{status:>22}] {step}{(' — ' + detail) if detail else ''}")


ledger = RunLedger("Gov Collect Agents", "agent", "T1")
token = graph_token()
print(f"run_id={ledger.run_id} dry_run={dry_run} registry={use_agent365_registry}")

registry_rows = []
identity_rows = []
blueprint_rows = []
dataverse_rows = []

## Source 1 — Agent 365 registry (preview)

Read-only by design: Graph exposes list and get only. Block / Unblock /
Delete / Reassign are UI-only, so those become 🟡 tasks, never writes.

In [ ]:
if use_agent365_registry:
    try:
        payload = graph_get(
            token, "/beta/admin/microsoft365Copilot/packages?$top=999"
        )
        registry_rows = shape_registry_agents(payload)
        ledger.count("registry", len(registry_rows))
        log("agent 365 registry", "Created", f"{len(registry_rows)} agents")
    except Exception as exc:  # noqa: BLE001
        # Not fatal: the whole point of the merge is that we degrade to the
        # sources that need no Agent 365 licence.
        ledger.error("registry", exc)
        log("agent 365 registry", "Skipped (no permission)", str(exc))
else:
    log("agent 365 registry", "Skipped (no permission)", "disabled by parameter")

## Source 2 — Entra Agent ID

In [ ]:
try:
    payload = graph_get(
        token,
        "/beta/servicePrincipals?$filter=servicePrincipalType eq 'AgentIdentity'"
        "&$top=999&$select=id,displayName,accountEnabled,createdDateTime",
    )
    identity_rows = shape_agent_identities(payload)
    ledger.count("identities", len(identity_rows))
    log("entra agent id", "Created", f"{len(identity_rows)} identities")
except Exception as exc:  # noqa: BLE001
    ledger.error("agentIdentities", exc)
    log("entra agent id", "Skipped (no permission)", str(exc))

try:
    payload = graph_get(
        token,
        "/beta/applications?$top=999&$select=id,displayName,signInAudience,requiredResourceAccess",
    )
    blueprint_rows = shape_blueprints(payload)
    ledger.count("gov_actual_agent_blueprints", len(blueprint_rows))
    log("blueprints", "Created", f"{len(blueprint_rows)} blueprints")
except Exception as exc:  # noqa: BLE001
    ledger.error("blueprints", exc)

## Source 3 — Dataverse `bot` table

Read from what the Power Platform collector already wrote, so this notebook
needs no Dataverse credentials of its own. It is also the only source that
distinguishes a **draft** agent.

In [ ]:
try:
    bots = spark.sql(  # noqa: F821
        f"SELECT environment_id, resource_id, resource_name, owner_name, created_at, state "
        f"FROM {lakehouse_name}.gov_actual_pp_resources WHERE resource_type = 'Agent'"
    ).collect()
    dataverse_rows = [
        {
            "agent_id": row["resource_id"],
            "name": row["resource_name"],
            "platform": "CopilotStudio",
            "source": "Dataverse",
            "state": row["state"] or "Draft",
            "owner_principal": row["owner_name"],
            "sponsor_principal": None,
            "blueprint_id": None,
            "agent_identity_id": None,
            "environment_id": row["environment_id"],
            "risk_flags_json": None,
            "created_at": row["created_at"],
        }
        for row in bots
    ]
    ledger.count("dataverse", len(dataverse_rows))
    log("dataverse bots", "Created", f"{len(dataverse_rows)} agents")
except Exception as exc:  # noqa: BLE001
    # Expected when the Power Platform collector has not run yet.
    ledger.error("dataverseBots", exc)
    log("dataverse bots", "Skipped (no permission)", str(exc))

## Merge

In [ ]:
agent_rows = merge_agents(registry_rows, identity_rows, dataverse_rows)
ledger.counts.pop("registry", None)
ledger.counts.pop("identities", None)
ledger.counts.pop("dataverse", None)
ledger.count("gov_actual_agents", len(agent_rows))

shadow = sum(1 for r in agent_rows if r.get("is_shadow") == "true")
ownerless = sum(1 for r in agent_rows if r.get("is_ownerless") == "true")
log(
    "merge",
    "Created",
    f"{len(agent_rows)} agents · {shadow} shadow · {ownerless} ownerless",
)

## Write

In [ ]:
TABLES = [
    ("gov_actual_agents", agent_rows),
    ("gov_actual_agent_blueprints", blueprint_rows),
]

for table, rows in TABLES:
    write_table(
        spark,  # noqa: F821
        lakehouse_name,
        table,
        stamp(rows, ledger.run_id),
        dry_run=dry_run,
        log=log,
    )

finish(ledger, spark, lakehouse_name, dry_run=dry_run)  # noqa: F821